# 01 - Entrainement ModCloth Fit Model V0

Objectif : entrainer le modele TensorFlow/Keras tabulaire de prediction du fit (`small`, `fit`, `large`) a partir du dataset ModCloth.

Ce notebook est limite au pipeline ModCloth V0 : recuperation Kaggle, inspection du dataset, appel du script `src.training.train_fit_model`, verification des artefacts, puis copie vers Google Drive.

Il ne travaille pas sur le CNN Fashion Product Images ni sur Polyvore.

## 1. Montage Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Clonage ou mise a jour du repo GitHub

Renseigne `REPO_URL` avec l'URL GitHub du projet, ou cree un secret Colab `FIT_OUTFIT_REPO_URL` contenant cette URL.

In [ ]:
import os
from pathlib import Path
from google.colab import userdata

DEFAULT_REPO_URL = 'https://github.com/<ton-compte-github>/fit-outfit-advisor.git'
REPO_URL = userdata.get('FIT_OUTFIT_REPO_URL') or DEFAULT_REPO_URL
REPO_DIR = Path('/content/fit-outfit-advisor')
BRANCH = 'main'

if '<ton-compte-github>' in REPO_URL:
    raise ValueError(
        'Configure REPO_URL dans cette cellule ou ajoute le secret Colab FIT_OUTFIT_REPO_URL.'
    )

if REPO_DIR.exists():
    print(f'Repo deja present, mise a jour : {REPO_DIR}')
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}
else:
    print(f'Clonage du repo : {REPO_URL}')
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Repertoire courant : {Path.cwd()}')

## 3. Installation des dependances

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt
!python -m pip install kaggle

## 4. Creation des dossiers temporaires dans `/content`

In [ ]:
CONTENT_ROOT = Path('/content/fit-outfit-runtime')
KAGGLE_DOWNLOAD_DIR = CONTENT_ROOT / 'kaggle_downloads'
CONTENT_DATA_DIR = CONTENT_ROOT / 'data'
CONTENT_ARTIFACT_DIR = CONTENT_ROOT / 'artifacts'

for directory in [CONTENT_ROOT, KAGGLE_DOWNLOAD_DIR, CONTENT_DATA_DIR, CONTENT_ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
    print(directory)

## 5. Chargement securise du secret Colab `KAGGLE_API_TOKEN`

Le secret peut contenir soit le JSON Kaggle complet (`{"username":"...","key":"..."}`), soit le format `username:key`. La valeur n'est jamais affichee.

In [ ]:
import json
import os

token = userdata.get('KAGGLE_API_TOKEN')
if not token:
    raise ValueError('Secret Colab KAGGLE_API_TOKEN absent.')

try:
    kaggle_credentials = json.loads(token)
except json.JSONDecodeError:
    if ':' not in token:
        raise ValueError(
            'KAGGLE_API_TOKEN doit contenir le JSON Kaggle ou le format username:key.'
        )
    username, key = token.split(':', 1)
    kaggle_credentials = {'username': username.strip(), 'key': key.strip()}

required_keys = {'username', 'key'}
if not required_keys <= set(kaggle_credentials):
    raise ValueError('KAGGLE_API_TOKEN doit fournir username et key.')

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json_path = kaggle_dir / 'kaggle.json'
kaggle_json_path.write_text(json.dumps(kaggle_credentials), encoding='utf-8')
os.chmod(kaggle_json_path, 0o600)

print('Configuration Kaggle OK.')

## 6. Telechargement du dataset ModCloth via Kaggle API

In [ ]:
KAGGLE_DATASET = 'rmisra/clothing-fit-dataset-for-size-recommendation'

!kaggle datasets download -d {KAGGLE_DATASET} -p {KAGGLE_DOWNLOAD_DIR} --unzip

downloaded_files = sorted(path for path in KAGGLE_DOWNLOAD_DIR.rglob('*') if path.is_file())
if not downloaded_files:
    raise FileNotFoundError('Aucun fichier telecharge depuis Kaggle. Verifie le token et le slug dataset.')

for path in downloaded_files:
    print(path)

## 7. Detection et affichage du fichier CSV ModCloth

Si Kaggle fournit ModCloth en JSON/JSONL, le notebook le convertit en CSV temporaire pour inspection et pour l'appel du script.

In [ ]:
import pandas as pd

csv_files = sorted(KAGGLE_DOWNLOAD_DIR.rglob('*.csv'))
modcloth_csv_files = [path for path in csv_files if 'modcloth' in path.name.lower()]

if modcloth_csv_files:
    DATASET_PATH = modcloth_csv_files[0]
    print(f'CSV ModCloth detecte : {DATASET_PATH}')
elif csv_files:
    DATASET_PATH = csv_files[0]
    print(f'CSV detecte : {DATASET_PATH}')
else:
    json_files = sorted(
        [*KAGGLE_DOWNLOAD_DIR.rglob('*.json'), *KAGGLE_DOWNLOAD_DIR.rglob('*.jsonl')]
    )
    modcloth_json_files = [path for path in json_files if 'modcloth' in path.name.lower()]
    if not modcloth_json_files:
        raise FileNotFoundError('Aucun CSV ni JSON ModCloth detecte dans le telechargement Kaggle.')

    source_json = modcloth_json_files[0]
    print(f'JSON ModCloth detecte : {source_json}')
    df_json = pd.read_json(source_json, lines=True)
    DATASET_PATH = CONTENT_DATA_DIR / 'modcloth_final_data.csv'
    df_json.to_csv(DATASET_PATH, index=False)
    print(f'CSV temporaire genere : {DATASET_PATH}')

print(f'DATASET_PATH = {DATASET_PATH}')

## 8. Inspection du dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)

print('df.shape =', df.shape)
print('\ndf.columns =')
print(list(df.columns))

display(df.head())

missing_values = df.isna().sum().sort_values(ascending=False)
print('\nValeurs manquantes par colonne :')
display(missing_values.to_frame('missing_count'))

## 9. Appel du script `src.training.train_fit_model`

Le notebook ne simule pas un entrainement : si `DATASET_PATH` est absent ou invalide, le script echouera explicitement.

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64

if not Path(DATASET_PATH).exists():
    raise FileNotFoundError(f'Dataset absent : {DATASET_PATH}')

os.chdir(REPO_DIR)
print(f'Repertoire courant : {Path.cwd()}')
!python -m src.training.train_fit_model --dataset "{DATASET_PATH}" --epochs {EPOCHS} --batch-size {BATCH_SIZE}

## 10. Metriques et artefacts generes

In [ ]:
artifact_paths = [
    REPO_DIR / 'models' / 'fit_model.keras',
    REPO_DIR / 'models' / 'encoders' / 'fit_preprocessor.joblib',
    REPO_DIR / 'models' / 'encoders' / 'fit_label_encoder.joblib',
    REPO_DIR / 'models' / 'encoders' / 'fit_metadata.json',
]

print('Artefacts attendus :')
for path in artifact_paths:
    status = 'OK' if path.exists() else 'ABSENT'
    size = path.stat().st_size if path.exists() else 0
    print(f'{status:6} {size:>12} bytes  {path}')

missing_artifacts = [path for path in artifact_paths if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError('Artefacts manquants apres entrainement : ' + ', '.join(map(str, missing_artifacts)))

metadata_path = REPO_DIR / 'models' / 'encoders' / 'fit_metadata.json'
print('\nMetadata fit :')
print(metadata_path.read_text(encoding='utf-8'))

## 11. Copie des artefacts finis vers Google Drive

In [ ]:
import shutil

DRIVE_ARTIFACT_DIR = Path('/content/drive/MyDrive/fit-outfit-advisor/artifacts/modcloth_fit')
DRIVE_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

for path in artifact_paths:
    destination = DRIVE_ARTIFACT_DIR / path.name
    shutil.copy2(path, destination)
    print(f'Copie : {path} -> {destination}')

print(f'\nArtefacts disponibles dans Google Drive : {DRIVE_ARTIFACT_DIR}')